# 07 - BAF LTN and Explanation Generalization

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/Tommyhuy1705/Explainable_NeuroSymbolic_Fraud_Detection.git"
KAGGLE_PROJECT_DIR = Path("/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection")

if KAGGLE:
    os.environ["THESIS_QUICK_RUN"] = "0"
    os.environ["THESIS_SYNTHETIC_FALLBACK"] = "0"
    if not KAGGLE_PROJECT_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(KAGGLE_PROJECT_DIR)],
            check=True,
        )

def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = [path for path in base.glob("**/configs") if path.is_dir()]
            if matches:
                return matches[0].parent
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"
INPUT_ROOTS = [OUTPUT_BASE, PROJECT_ROOT / "results/runs/notebooks"]
if Path("/kaggle/input").exists():
    INPUT_ROOTS.append(Path("/kaggle/input"))

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_COMMIT = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print({
    "project_root": str(PROJECT_ROOT),
    "git_commit": GIT_COMMIT,
    "quick_run": QUICK_RUN,
    "synthetic_fallback": ALLOW_SYNTHETIC_FALLBACK,
    "kaggle": KAGGLE,
})

Cloning into '/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection'...


{'project_root': '/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection', 'git_commit': '0f8234cb38cff4871ead148944284527e43be5c9', 'quick_run': False, 'synthetic_fallback': False, 'kaggle': True}


## Thiết lập

Notebook chỉ đọc frozen BAF reference predictor từ Notebook 03. Rules fit trên train months 0-4,
được kiểm tra trên validation month 5 và locked test months 6-7.

In [2]:
# EXPECTED_DATASET = "ieee_cis"
EXPECTED_DATASET = "baf"

manifest_paths = list(
    Path("/kaggle/input").glob("**/frozen_reference_manifest.json")
)

matching_manifests = []

for manifest_path in manifest_paths:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

    if manifest.get("dataset_name") == EXPECTED_DATASET:
        artifact_path = manifest_path.parent / manifest["artifact_file"]

        print({
            "dataset": manifest.get("dataset_name"),
            "model": manifest.get("model"),
            "reference_seed": manifest.get("reference_seed"),
            "quick_run": manifest.get("quick_run"),
            "manifest": str(manifest_path),
            "artifact": str(artifact_path),
            "artifact_exists": artifact_path.exists(),
        })

        if artifact_path.exists():
            matching_manifests.append(manifest_path)

assert len(matching_manifests) == 1, (
    f"Expected exactly one valid artifact for {EXPECTED_DATASET}, "
    f"found {len(matching_manifests)}. Check Kaggle Add Input."
)

print(f"Artifact preflight passed for {EXPECTED_DATASET}.")

{'dataset': 'baf', 'model': 'lightgbm', 'reference_seed': 42, 'quick_run': False, 'manifest': '/kaggle/input/notebooks/giahuytranviet/03-baf-model-benchmarks/thesis_outputs/03_baf_model_benchmarks/frozen_reference_manifest.json', 'artifact': '/kaggle/input/notebooks/giahuytranviet/03-baf-model-benchmarks/thesis_outputs/03_baf_model_benchmarks/frozen_reference_artifact.npz', 'artifact_exists': True}
Artifact preflight passed for baf.


In [3]:
from src.artifacts import assert_frozen_alignment, load_frozen_reference_artifact
from src.data import load_config, prepare_dataset
from src.experiment import load_experiment_data

config = load_config(PROJECT_ROOT / "configs/baf.yaml")
frame, data_source = load_experiment_data(
    config, max_rows=12000 if QUICK_RUN else None,
    synthetic_fallback=ALLOW_SYNTHETIC_FALLBACK,
    synthetic_rows=12000 if QUICK_RUN else 6000,
)
prepared = prepare_dataset(frame, config)
artifact = load_frozen_reference_artifact(
    "baf", expected_config=config,
    search_roots=[OUTPUT_BASE / "03_baf_model_benchmarks", *INPUT_ROOTS],
)
assert_frozen_alignment(artifact, prepared.y_validation, prepared.y_test)
if bool(artifact["manifest"]["quick_run"]) != QUICK_RUN:
    raise ValueError("Notebook mode and frozen artifact quick_run flag do not match")
probabilities = artifact["test_probability"]
threshold = float(artifact["manifest"]["threshold"])
print({"data_source": data_source, "manifest": artifact["manifest"]})

{'data_source': 'real', 'manifest': {'artifact_schema_version': 1, 'dataset_name': 'baf', 'data_source': 'real', 'configured_files': ['Base.csv'], 'model_key': 'tree', 'model': 'lightgbm', 'selection_basis': 'highest mean validation raw PR-AUC', 'reference_seed': 42, 'threshold': 0.07246376811594203, 'calibration_method': 'isotonic', 'config_sha256': '3321472443ae5209b6bf865f682ee56a28d6924d88015eb44178349af1fefd94', 'split_summary': [{'split': 'train', 'rows': 675666, 'time_min': 0, 'time_max': 4, 'time_group_count': 5, 'time_groups': [0, 1, 2, 3, 4], 'overlap_groups': [], 'group_disjoint': True, 'fraud_rate': 0.009975342846909568}, {'split': 'validation', 'rows': 119323, 'time_min': 5, 'time_max': 5, 'time_group_count': 1, 'time_groups': [5], 'overlap_groups': [], 'group_disjoint': True, 'fraud_rate': 0.01182504630289215}, {'split': 'test', 'rows': 205011, 'time_min': 6, 'time_max': 7, 'time_group_count': 2, 'time_groups': [6, 7], 'overlap_groups': [], 'group_disjoint': True, 'fraud_

## Rule and explanation results

In [4]:
from src.explanation import (
    RuleExplainer, bootstrap_explanation_precision_gain,
    explanation_quality_metrics, rule_quality_table,
)
from src.logic import FraudKnowledgeBase, FraudRuleEngine

output_dir = OUTPUT_BASE / "07_baf_ltn_generalization"
output_dir.mkdir(parents=True, exist_ok=True)
target = config["dataset"]["target_column"]
engine = FraudRuleEngine(config["logic"]["rules"]).fit(prepared.train_frame, target)
knowledge_base = FraudKnowledgeBase(engine)
validation_truth = engine.evaluate(prepared.validation_frame)
test_truth = engine.evaluate(prepared.test_frame)
activation = float(config["logic"]["activation_threshold"])
validation_quality = rule_quality_table(validation_truth, prepared.y_validation, activation).assign(split="validation")
test_quality = rule_quality_table(test_truth, prepared.y_test, activation).assign(split="test")
rule_quality = pd.concat([validation_quality, test_quality], ignore_index=True)
satisfaction = pd.DataFrame([
    {"split": "train", **knowledge_base.satisfaction_breakdown(prepared.train_frame, target)},
    {"split": "validation", **knowledge_base.satisfaction_breakdown(prepared.validation_frame, target)},
    {"split": "test", **knowledge_base.satisfaction_breakdown(prepared.test_frame, target)},
])
explainer = RuleExplainer(engine, activation, config["logic"]["top_k_rules"])
explanations = explainer.explain(prepared.test_frame, probabilities, threshold)
quality = explanation_quality_metrics(explanations, prepared.y_test, probabilities, threshold)
quality.update(bootstrap_explanation_precision_gain(
    explanations, prepared.y_test, probabilities, threshold,
    n_bootstrap=config["evaluation"]["bootstrap_iterations"], seed=config["project"]["seed"],
))
explanation_quality = pd.DataFrame([quality])
display(rule_quality.round(4), satisfaction.round(4), explanation_quality.round(4))
rule_quality.to_csv(output_dir / "baf_rule_quality.csv", index=False)
satisfaction.to_csv(output_dir / "baf_knowledge_base_satisfaction.csv", index=False)
explanation_quality.to_csv(output_dir / "baf_explanation_quality.csv", index=False)

,rule,coverage,active_count,fraud_precision,lift,mean_truth,rule_auc,split
0,foreign_high_limit_request,0.0003,37,0.1892,15.9990,0.0024,0.5145,validation
1,device_email_linkage,0.0151,1799,0.0650,5.4999,0.5054,0.5341,validation
2,short_session_high_risk,0.0225,2682,0.0246,2.0810,0.0687,0.4979,validation
3,high_velocity,0.0041,486,0.0082,0.6960,0.0721,0.4702,validation
4,young_high_income_request,0.0000,0,0.0000,0.0000,0.2355,0.6194,validation
5,foreign_high_limit_request,0.0004,74,0.2432,17.3272,0.0024,0.5137,test
6,device_email_linkage,0.0133,2719,0.0585,4.1656,0.5044,0.5195,test
7,short_session_high_risk,0.0199,4070,0.0334,2.3803,0.0636,0.4947,test
8,high_velocity,0.0043,879,0.0114,0.8104,0.0482,0.4729,test
9,young_high_income_request,0.0000,0,0.0000,0.0000,0.2620,0.6102,test


,split,overall_satisfaction,positive_satisfaction,negative_satisfaction,balanced_satisfaction
0,train,0.4788,0.5655,0.4780,0.5217
1,validation,0.4879,0.5542,0.4872,0.5207
2,test,0.4895,0.5406,0.4887,0.5146


,explanation_coverage_all,explanation_coverage_alerts,mean_rule_count,sparsity,prediction_rule_consistency,contradiction_rate,explained_alert_precision,all_alert_precision,explained_alert_precision_gain,precision_gain_bootstrap_mean,precision_gain_ci_low,precision_gain_ci_high,bootstrap_valid_iterations
0,0.0375,0.1151,0.0378,0.9636,0.9344,0.0656,0.223,0.1703,0.0527,0.0525,0.0276,0.0777,1000.0


In [5]:
stability = validation_quality.merge(test_quality, on="rule", suffixes=("_validation", "_test"))
stability["coverage_delta"] = stability["coverage_test"] - stability["coverage_validation"]
stability["lift_delta"] = stability["lift_test"] - stability["lift_validation"]
display(stability[["rule", "coverage_delta", "lift_delta"]].round(4))
stability.to_csv(output_dir / "baf_rule_stability.csv", index=False)

,rule,coverage_delta,lift_delta
0,foreign_high_limit_request,0.0001,1.3281
1,device_email_linkage,-0.0018,-1.3343
2,short_session_high_risk,-0.0026,0.2992
3,high_velocity,0.0002,0.1144
4,young_high_income_request,0.0000,0.0000


## Takeaways

In [6]:
best_rule = test_quality.sort_values("lift", ascending=False).iloc[0]
display(Markdown(
    f"- Frozen predictor: **{artifact['manifest']['model']}**, seed **{artifact['manifest']['reference_seed']}**.\n"
    f"- Highest BAF test rule lift: **{best_rule['rule']} = {best_rule['lift']:.3f}**.\n"
    f"- Alert explanation coverage: **{quality['explanation_coverage_alerts']:.3f}**.\n"
    "- Cross-dataset evidence is bounded to these public benchmark distributions."
))

- Frozen predictor: **lightgbm**, seed **42**.
- Highest BAF test rule lift: **foreign_high_limit_request = 17.327**.
- Alert explanation coverage: **0.115**.
- Cross-dataset evidence is bounded to these public benchmark distributions.